# Day-01: Linear Regression
Complete implementation of Linear Regression from scratch and using scikit-learn

## 1. Setup: Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'seaborn', 'joblib']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
print('All dependencies installed successfully!')

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from joblib import dump, load

# Set random seed for reproducibility
np.random.seed(42)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries imported successfully!')

## 3. Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv('dataset.csv')

print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
print('\nFirst few rows:')
print(df.head())
print('\nDataset Info:')
print(df.info())
print('\nBasic Statistics:')
print(df.describe())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(df['x'], df['y'], alpha=0.6, s=50)
axes[0].set_xlabel('x (Feature)', fontsize=12)
axes[0].set_ylabel('y (Target)', fontsize=12)
axes[0].set_title('Feature vs Target', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Correlation heatmap
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', ax=axes[1], cbar_kws={'label': 'Correlation'})
axes[1].set_title('Correlation Matrix', fontsize=12)

plt.tight_layout()
plt.show()

print('Correlation with target:')
print(correlation['y'].sort_values(ascending=False))

## 5. Prepare Data for Training

In [ ]:
# Extract features and target
X = df.drop('y', axis=1).values  # Features
y = df['y'].values  # Target

# Add bias term (column of 1s)
X_bias = np.column_stack([np.ones(X.shape[0]), X])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_bias = np.column_stack([np.ones(X_train.shape[0]), X_train])
X_test_bias = np.column_stack([np.ones(X_test.shape[0]), X_test])

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size: {X_test.shape[0]}')
print(f'Features (with bias): {X_train_bias.shape[1]}')

## 6. Implement Linear Regression from Scratch

In [ ]:
class LinearRegressionCustom:
    def __init__(self):
        self.theta = None
        self.loss_history = []
    
    def normal_equation(self, X, y):
        """
        Solve using normal equation: θ = (X^T X)^(-1) X^T y
        """
        self.theta = np.linalg.inv(X.T @ X) @ X.T @ y
        return self.theta
    
    def gradient_descent(self, X, y, alpha=0.01, epochs=1000):
        """
        Train using gradient descent: θ := θ - α ∇J(θ)
        """
        m = X.shape[0]
        self.theta = np.zeros(X.shape[1])
        self.loss_history = []
        
        for epoch in range(epochs):
            # Predictions
            y_pred = X @ self.theta
            
            # Cost (MSE)
            cost = (1 / (2 * m)) * np.sum((y_pred - y) ** 2)
            self.loss_history.append(cost)
            
            # Gradient
            gradient = (1 / m) * X.T @ (y_pred - y)
            
            # Update theta
            self.theta = self.theta - alpha * gradient
        
        return self.theta, self.loss_history
    
    def predict(self, X):
        return X @ self.theta
    
    def save_model(self, path):
        dump(self.theta, path)
    
    def load_model(self, path):
        self.theta = load(path)

# Functions for evaluation
def mean_squared_error_custom(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score_custom(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

print('Custom Linear Regression class defined!')

## 7. Train Using Normal Equation

In [ ]:
# Train using normal equation
model_ne = LinearRegressionCustom()
theta_ne = model_ne.normal_equation(X_train_bias, y_train)

# Predictions
y_pred_train_ne = model_ne.predict(X_train_bias)
y_pred_test_ne = model_ne.predict(X_test_bias)

# Evaluation
mse_train_ne = mean_squared_error_custom(y_train, y_pred_train_ne)
mse_test_ne = mean_squared_error_custom(y_test, y_pred_test_ne)
r2_train_ne = r2_score_custom(y_train, y_pred_train_ne)
r2_test_ne = r2_score_custom(y_test, y_pred_test_ne)

print('=== Normal Equation Results ===')
print(f'Coefficients (θ): {theta_ne}')
print(f'Training MSE: {mse_train_ne:.4f}')
print(f'Testing MSE: {mse_test_ne:.4f}')
print(f'Training R²: {r2_train_ne:.4f}')
print(f'Testing R²: {r2_test_ne:.4f}')

## 8. Train Using Gradient Descent

In [ ]:
# Train using gradient descent
model_gd = LinearRegressionCustom()
theta_gd, loss_history = model_gd.gradient_descent(X_train_bias, y_train, alpha=0.01, epochs=1000)

# Predictions
y_pred_train_gd = model_gd.predict(X_train_bias)
y_pred_test_gd = model_gd.predict(X_test_bias)

# Evaluation
mse_train_gd = mean_squared_error_custom(y_train, y_pred_train_gd)
mse_test_gd = mean_squared_error_custom(y_test, y_pred_test_gd)
r2_train_gd = r2_score_custom(y_train, y_pred_train_gd)
r2_test_gd = r2_score_custom(y_test, y_pred_test_gd)

print('=== Gradient Descent Results ===')
print(f'Coefficients (θ): {theta_gd}')
print(f'Training MSE: {mse_train_gd:.4f}')
print(f'Testing MSE: {mse_test_gd:.4f}')
print(f'Training R²: {r2_train_gd:.4f}')
print(f'Testing R²: {r2_test_gd:.4f}')

# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Gradient Descent: Loss Over Epochs')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Compare with Scikit-learn

In [ ]:
# Train scikit-learn model
model_sklearn = LinearRegression()
model_sklearn.fit(X_train, y_train)

# Predictions
y_pred_train_sk = model_sklearn.predict(X_train)
y_pred_test_sk = model_sklearn.predict(X_test)

# Evaluation
mse_train_sk = mean_squared_error(y_train, y_pred_train_sk)
mse_test_sk = mean_squared_error(y_test, y_pred_test_sk)
r2_train_sk = r2_score(y_train, y_pred_train_sk)
r2_test_sk = r2_score(y_test, y_pred_test_sk)

print('=== Scikit-learn LinearRegression Results ===')
print(f'Intercept: {model_sklearn.intercept_:.4f}')
print(f'Coefficients: {model_sklearn.coef_}')
print(f'Training MSE: {mse_train_sk:.4f}')
print(f'Testing MSE: {mse_test_sk:.4f}')
print(f'Training R²: {r2_train_sk:.4f}')
print(f'Testing R²: {r2_test_sk:.4f}')

## 10. Visualization: Fit, Residuals, and Comparison

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted (Normal Equation)
axes[0, 0].scatter(y_test, y_pred_test_ne, alpha=0.6)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual y')
axes[0, 0].set_ylabel('Predicted y')
axes[0, 0].set_title('Normal Equation: Actual vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals (Normal Equation)
residuals_ne = y_test - y_pred_test_ne
axes[0, 1].scatter(y_pred_test_ne, residuals_ne, alpha=0.6)
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted y')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Normal Equation: Residuals')
axes[0, 1].grid(True, alpha=0.3)

# 3. Actual vs Predicted (Gradient Descent)
axes[1, 0].scatter(y_test, y_pred_test_gd, alpha=0.6, color='green')
axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1, 0].set_xlabel('Actual y')
axes[1, 0].set_ylabel('Predicted y')
axes[1, 0].set_title('Gradient Descent: Actual vs Predicted')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residuals (Gradient Descent)
residuals_gd = y_test - y_pred_test_gd
axes[1, 1].scatter(y_pred_test_gd, residuals_gd, alpha=0.6, color='green')
axes[1, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 1].set_xlabel('Predicted y')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Gradient Descent: Residuals')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Model Comparison Summary

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Method': ['Normal Equation', 'Gradient Descent', 'Scikit-learn'],
    'Train MSE': [mse_train_ne, mse_train_gd, mse_train_sk],
    'Test MSE': [mse_test_ne, mse_test_gd, mse_test_sk],
    'Train R²': [r2_train_ne, r2_train_gd, r2_train_sk],
    'Test R²': [r2_test_ne, r2_test_gd, r2_test_sk]
})

print('\n=== Model Comparison ===')
print(comparison.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(comparison))
width = 0.35

axes[0].bar(x_pos - width/2, comparison['Train MSE'], width, label='Train MSE')
axes[0].bar(x_pos + width/2, comparison['Test MSE'], width, label='Test MSE')
axes[0].set_ylabel('MSE')
axes[0].set_title('Model Comparison: MSE')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison['Method'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x_pos - width/2, comparison['Train R²'], width, label='Train R²')
axes[1].bar(x_pos + width/2, comparison['Test R²'], width, label='Test R²')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Model Comparison: R² Score')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison['Method'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 12. Save Model

In [ ]:
# Save models
model_ne.save_model('linear_regression_ne.pkl')
print('Normal Equation model saved as: linear_regression_ne.pkl')

model_gd.save_model('linear_regression_gd.pkl')
print('Gradient Descent model saved as: linear_regression_gd.pkl')

dump(model_sklearn, 'linear_regression_sklearn.pkl')
print('Scikit-learn model saved as: linear_regression_sklearn.pkl')

## 13. Unit Tests and Validation

In [ ]:
# Unit tests
print('=== Unit Tests ===')

# Test 1: Predictions shape
assert y_pred_test_ne.shape == y_test.shape, f'Shape mismatch: {y_pred_test_ne.shape} != {y_test.shape}'
print('✓ Test 1: Prediction shape matches target shape')

# Test 2: Coefficients shape
assert theta_ne.shape[0] == X_train_bias.shape[1], f'Theta shape mismatch'
print('✓ Test 2: Coefficients shape matches features (with bias)')

# Test 3: MSE is non-negative
assert mse_test_ne >= 0, 'MSE should be non-negative'
print('✓ Test 3: MSE is non-negative')

# Test 4: R² is between -inf and 1
assert r2_test_ne <= 1, 'R² should be <= 1'
print('✓ Test 4: R² is valid')

# Test 5: Model predictions are numeric
assert np.all(np.isfinite(y_pred_test_ne)), 'Predictions contain non-finite values'
print('✓ Test 5: All predictions are finite')

# Test 6: Models produce reasonable results
assert abs(r2_test_ne - r2_test_gd) < 0.01, 'Normal Equation and Gradient Descent should give similar results'
print('✓ Test 6: Normal Equation and Gradient Descent produce similar results')

print('\n✓ All tests passed!')